In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
#!git clone https://github.com/recsyspolimi/RecSys_Course_AT_PoliMi

import os

"""current_directory = os.getcwd()
print(f"Current directory: {current_directory}")
if current_directory.endswith("RecSys_Course_AT_PoliMi"):
    pass
else:
    os.chdir("/RecSys_Course_AT_PoliMi")
"""
!pwd
os.chdir("RecSys_Course_AT_PoliMi")
!python3 run_compile_all_cython.py

/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge
run_compile_all_cython: Found 11 Cython files in 5 folders...
run_compile_all_cython: All files will be compiled using your current python environment: '/usr/local/bin/python3'
Compiling [1/11]: MatrixFactorizationImpressions_Cython_Epoch.pyx... 
/Library/Frameworks/Python.framework/Versions/3.13/Resources/Python.app/Contents/MacOS/Python: can't open file '/Users/filippo/Documents/PoliMI/Recommender': [Errno 2] No such file or directory
Traceback (most recent call last):
  File "/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/RecSys_Course_AT_PoliMi/run_compile_all_cython.py", line 60, in <module>
    run_compile_subprocess(file_path, [file_name])
    ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/RecSys_Course_AT_PoliMi/CythonCompiler/run_compile_subprocess.py", line 51, in run_compile_subprocess
    raise exc
  File "/Users/filippo/Document

In [3]:
import os
import time 
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse as sps
import matplotlib.pyplot as pyplot
%matplotlib inline

from sklearn.model_selection import KFold
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from skopt.space import Real, Integer, Categorical
from Evaluation.Evaluator import EvaluatorHoldout
from HyperparameterTuning.SearchBayesianSkopt import SearchBayesianSkopt
from Recommenders.MatrixFactorization.SVDFeatureRecommender import SVDFeature

Tensorflow is not available


In [4]:
df_train = pd.read_csv("data_train.csv")
df_test_user = pd.read_csv("data_target_users_test.csv")

In [5]:
def split_train_in_five_percentage_global_sample(URM_all, train_percentages):
    """
    The function splits an URM in five matrices based on provided percentages.
    :param URM_all: The full URM matrix
    :param train_percentages: A list of percentages (must sum to 1.0)
    :return: A list of 5 sparse matrices
    """

    import numpy as np
    from scipy.sparse import coo_matrix
    from Data_manager.IncrementalSparseMatrix import IncrementalSparseMatrix

    assert len(train_percentages) == 5, "You must provide exactly 5 percentages."
    assert abs(sum(train_percentages) - 1.0) < 1e-6, "Percentages must sum to 1.0."

    num_users, num_items = URM_all.shape

    # Builders for each of the 5 matrices
    builders = [
        IncrementalSparseMatrix(n_rows=num_users, n_cols=num_items, auto_create_col_mapper=False, auto_create_row_mapper=False)
        for _ in range(5)
    ]

    URM_all_coo = coo_matrix(URM_all)

    # Shuffle indices
    indices_for_sampling = np.arange(URM_all.nnz, dtype=np.int32)
    np.random.shuffle(indices_for_sampling)

    # Calculate the number of interactions for each split
    split_sizes = [int(URM_all.nnz * percentage) for percentage in train_percentages]
    cumulative_sizes = np.cumsum(split_sizes)

    # Divide the indices into 5 groups
    indices_splits = [
        indices_for_sampling[cumulative_sizes[i - 1]:cumulative_sizes[i]] if i > 0 else indices_for_sampling[:cumulative_sizes[i]]
        for i in range(5)
    ]

    # Populate the builders
    for i, builder in enumerate(builders):
        builder.add_data_lists(
            URM_all_coo.row[indices_splits[i]],
            URM_all_coo.col[indices_splits[i]],
            URM_all_coo.data[indices_splits[i]],
        )

    # Convert to sparse matrices
    sparse_matrices = [builder.get_SparseMatrix() for builder in builders]

    # Ensure all outputs are in csr_matrix format
    sparse_matrices = [sp.csr_matrix(matrix) for matrix in sparse_matrices]

    return sparse_matrices

In [6]:
from scipy.sparse import coo_matrix

#valore 1 per ogni coppia (row, col)
data = [1] * len(df_train)
df_train["row"] = df_train["row"].astype(int)
df_train["col"] = df_train["col"].astype(int)

# matrice COO
URM_all = sp.csr_matrix((data, (df_train["row"], df_train["col"])))

In [7]:
train_percentages = [0.2, 0.2, 0.2, 0.2, 0.2]  # Cinque parti uguali

URM_parts = split_train_in_five_percentage_global_sample(URM_all, train_percentages)
URM_parts

[<Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>]

In [8]:

import time 

class SaveResults(object):
    
    def __init__(self):
        self.results_df = pd.DataFrame(columns=["result", "train_time (min)"])
    
    def __call__(self, optuna_study, optuna_trial):
        hyperparam_dict = optuna_trial.params.copy()
        hyperparam_dict["result"] = optuna_trial.values[0]
        
        # Retrieve the optimal number of epochs and training time from the "user attributes" of the trial
        #hyperparam_dict["epochs"] = optuna_trial.user_attrs["epochs"]
        hyperparam_dict["train_time (min)"] = optuna_trial.user_attrs["train_time (min)"]
        
        self.results_df.loc[len(self.results_df)] = hyperparam_dict
        
        
import os
import shutil

def objective_function_svd_feature(optuna_trial):
    start_time = time.time()
    scores = []
    
    # Parametri suggeriti da Optuna
    num_factors = optuna_trial.suggest_int("num_factors", 8, 256)
    learning_rate = optuna_trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True)
    epochs = optuna_trial.suggest_int("epochs", 10, 100)
    
    # Regolarizzazioni (molto importanti per evitare overfitting)
    user_reg = optuna_trial.suggest_float("user_reg", 1e-5, 1e-1, log=True)
    item_reg = optuna_trial.suggest_float("item_reg", 1e-5, 1e-1, log=True)
    user_bias_reg = optuna_trial.suggest_float("user_bias_reg", 1e-5, 1e-1, log=True)
    item_bias_reg = optuna_trial.suggest_float("item_bias_reg", 1e-5, 1e-1, log=True)

    for i in range(5):
        URM_combined = sum(URM_parts[j] for j in range(len(URM_parts)) if j != i)
        
        # Creiamo una cartella temporanea unica per questo trial e fold
        temp_folder = f"./result_experiments/SVDFeature_temp_trial_{optuna_trial.number}_fold_{i}/"
        
        # Inizializzazione SVDFeature (puoi passare ICM o UCM se li hai)
        recommender_instance = SVDFeature(URM_combined, ICM=None, UCM=None)

        recommender_instance.fit(
            num_factors = num_factors,
            learning_rate = learning_rate,
            epochs = epochs,
            user_reg = user_reg,
            item_reg = item_reg,
            user_bias_reg = user_bias_reg,
            item_bias_reg = item_bias_reg,
            temp_file_folder = temp_folder
        )
        
        # Valutazione
        evaluator_test = EvaluatorHoldout(URM_parts[i], cutoff_list=[20])
        result, _ = evaluator_test.evaluateRecommender(recommender_instance)
        
        scores.append(result.loc[20]["RECALL"])
        
        # Pulizia: rimuoviamo la cartella temporanea dopo ogni fold per non riempire il disco
        shutil.rmtree(temp_folder, ignore_errors=True)
        
    optuna_trial.set_user_attr("train_time (min)", (time.time() - start_time)/60) 
    print(f"Trial {optuna_trial.number} - Scores: {scores}")
    
    return sum(scores) / len(scores)

In [9]:
import optuna
optuna_study = optuna.create_study(direction="maximize")
        
save_results = SaveResults()
        
optuna_study.optimize(objective_function_svd_feature,
                      callbacks=[save_results],
                      n_trials = 200)

[I 2025-12-28 21:08:24,858] A new study created in memory with name: no-name-bb126adc-1934-437c-b4b7-11bcacf3d485


SVDFeature: Using Temp folder './result_experiments/SVDFeature_temp_trial_0_fold_0/'
SVDFeature: Writing input file in feature format


100%|██████████| 2434444/2434444 [00:18<00:00, 131289.23it/s]
[W 2025-12-28 21:08:43,483] Trial 0 failed with parameters: {'num_factors': 71, 'learning_rate': 0.06458527713035694, 'epochs': 64, 'user_reg': 0.00020055107367256654, 'item_reg': 0.034483238899871996, 'user_bias_reg': 0.018044450690309292, 'item_bias_reg': 0.0012899102145078193} because of the following error: FileNotFoundError(2, 'No such file or directory').
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/wp/jydg89697jzcwnllv2_yz6d40000gn/T/ipykernel_2053/3859881869.py", line 46, in objective_function_svd_feature
    recommender_instance.fit(
    ~~~~~~~~~~~~~~~~~~~~~~~~^
        num_factors = num_factors,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<6 lines>...
        temp_file_folder = temp_folder
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )


SVDFeature: Fit starting


FileNotFoundError: [Errno 2] No such file or directory: 'svd_feature'

In [ ]:
optuna_study.best_trial.params

In [ ]:
import os

current_directory = os.getcwd()
if current_directory.endswith("Results"):
    print("In the correct directory.")
    pass
else:
    os.chdir("/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/Models/Results")

# 1. Get the dataframe from the study
df_trials = optuna_study.trials_dataframe()

# 2. Filter for the columns specific to ScaledPureSVDRecommender
try:
    df_filtered = df_trials[[
        'number', 
        'params_num_factors',
        'params_scaling_items', 
        'params_scaling_users',
        'value'
    ]].copy()
except KeyError as e:
    print(f"Error: One or more parameters not found in study. Check your objective function names. {e}")
    # Fallback if you haven't run the study yet or names differ
    df_filtered = df_trials.copy() 

# 3. Rename the columns to be CSV-friendly
df_filtered.columns = ['trial_id', 'num_factors', 'scaling_items', 'scaling_users', 'recall']

# 4. Automatic Versioning Logic
base_filename = "scaledpure_svd_results_v"
extension = ".csv"
version = 1

while True:
    csv_filename = f"{base_filename}{version}{extension}"
    if not os.path.exists(csv_filename):
        break # Trovato un nome file non ancora esistente
    version += 1

# 5. Save to CSV (Senza ordinamento)
df_filtered.to_csv(csv_filename, index=False)

print(f"✅ Full results saved to {csv_filename}")

# 6. Visualizza i Top 5 (opzionale, solo per visualizzazione rapida)
print("------------ FIRST 5 TRIALS ------")
print(df_filtered.head(5))
print("----------------------------------")

In [ ]:
#/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/Models

#!git clone https://github.com/recsyspolimi/RecSys_Course_AT_PoliMi

import os

current_directory = os.getcwd()
if current_directory.endswith("Results"):
    print("In the correct directory.")
    pass
else:
    os.chdir("/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/Models/Results")

df_filtered.to_csv("scaledpureSVD_opt_results.csv", index=False)

!pwd
!python3 run_compile_all_cython.py

In [ ]:
optuna_study.best_value
best_index = save_results.results_df["result"].idxmax()
best_hyperparams = save_results.results_df.loc[best_index].to_dict()

del best_hyperparams["result"]
del best_hyperparams["train_time (min)"]
print("-----------best hyperparameters-----------\n")
print(best_hyperparams)